In [ ]:

# ============================================================
# 1. Data import + 2x2 binning + min-max normalization
#         + mean centering
#
# Experimental design:
#   3 independent experiments
#   Each experiment contains 3 groups
#   Each group contains 10 cells
#   Total: 90 cells
#
# Folder structure:
# 250826_FLAG/
# ├── Experiment_1/
# │   ├── SNCA/
# │   ├── UBL3/
# │   └── CAAX/
# ├── Experiment_2/
# │   ├── SNCA/
# │   ├── UBL3/
# │   └── CAAX/
# └── Experiment_3/
#     ├── SNCA/
#     ├── UBL3/
#     └── CAAX/
# ============================================================

import os
import re
import numpy as np
import pandas as pd
from scipy.stats import zscore
from sklearn.preprocessing import MinMaxScaler


# ------------------------------------------------------------
# 0.1 Basic settings
# ------------------------------------------------------------

direct = 'AAV'

root_path = f'/{direct}/'

# Output directory
path = os.path.join(root_path, 'result_mmCen/')
os.makedirs(path, exist_ok=True)

# Three independent experiments
experiments = ['e1', 'e2', 'e3']

# Three experimental groups
groups = ['SNCA', 'UBL3', 'CAAX']

# Expected number of cells in each group of each experiment
expected_cells = 10


# ------------------------------------------------------------
# 0.2 Min-max normalization function
# ------------------------------------------------------------

def group_minmax(x):
    mn = x.min()
    mx = x.max()

    if mn == mx:
        return x * 0.0
    else:
        return (x - mn) / (mx - mn)


# ------------------------------------------------------------
# 0.3 Read and process one group folder
# ------------------------------------------------------------

def read_csv(folder_path, name='syn',
             norm_method='normalized',
             experiment_id=None,
             group_name=None):

    combined_df = pd.DataFrame()

    csv_files = sorted([
        f for f in os.listdir(folder_path)
        if f.lower().endswith('.csv')
    ])

    if len(csv_files) == 0:
        raise ValueError(
            f'No CSV files found in: {folder_path}'
        )

    for file in csv_files:

        file_path = os.path.join(folder_path, file)

        df = pd.read_csv(
            file_path,
            usecols=lambda c: c in ('X', 'Y', 'Value', 'Red')
        )

        df.columns = ['x', 'y', name]

        # ----------------------------------------------------
        # Original cell label extraction
        # ----------------------------------------------------

        original_label = os.path.splitext(file)[0]

        # Try the original filename pattern first
        match = re.search(
            r'Image(\d+)_ch03(-\d+)',
            original_label
        )

        if match:
            raw_cell_id = ''.join(match.groups())
        else:
            # If filename does not match the original pattern,
            # use the complete filename (without extension)
            raw_cell_id = original_label

        # Create globally unique cell ID
        cell_id = (
            f'{experiment_id}_{group_name}_{raw_cell_id}'
        )

        df['label'] = original_label
        df['label1_raw'] = raw_cell_id

        # IMPORTANT:
        # label1 is now globally unique across experiments
        # and groups. Downstream cells can still use label1.
        df['label1'] = cell_id

        df['Cell_ID'] = cell_id
        df['Experiment_ID'] = experiment_id
        df['Group'] = group_name

        # ----------------------------------------------------
        # Original 2x2 binning
        # ----------------------------------------------------

        df['x_idx'] = df['x'] - df['x'].min()
        df['y_idx'] = df['y'] - df['y'].min()

        df['x_bin'] = df['x_idx'] // 2
        df['y_bin'] = df['y_idx'] // 2

        df[f'{name}_bin'] = (
            df.groupby(
                ['label1', 'x_bin', 'y_bin']
            )[name].transform('mean')
        )

        combined_df = pd.concat(
            [combined_df, df],
            axis=0,
            ignore_index=True
        )

    # --------------------------------------------------------
    # Original per-cell pixel and intensity information
    # --------------------------------------------------------

    pixel = combined_df.groupby('label1').size()

    totalInt = (
        combined_df.groupby('label1')[name].sum()
    )

    minv = combined_df.groupby('label1')[name].min()
    maxv = combined_df.groupby('label1')[name].max()

    combined_df['pixel_number'] = (
        combined_df['label1'].map(pixel)
    )

    combined_df['total_int'] = (
        combined_df['label1'].map(totalInt)
    )

    combined_df[f'relative_{name}'] = (
        combined_df[name] / combined_df['pixel_number']
    )

    # --------------------------------------------------------
    # Original min-max normalization
    # --------------------------------------------------------

    combined_df[f'min_{name}'] = (
        combined_df['label1'].map(minv)
    )

    combined_df[f'max_{name}'] = (
        combined_df['label1'].map(maxv)
    )

    intensity_range = (
        combined_df[f'max_{name}']
        - combined_df[f'min_{name}']
    )

    combined_df[f'maxnorm_{name}'] = np.where(
        intensity_range != 0,
        (
            combined_df[name]
            - combined_df[f'min_{name}']
        ) / intensity_range,
        0.0
    )

    # --------------------------------------------------------
    # Original mean centering
    # --------------------------------------------------------

    meanv = (
        combined_df.groupby('label1')[
            f'maxnorm_{name}'
        ].mean()
    )

    combined_df[f'mean_{name}'] = (
        combined_df['label1'].map(meanv)
    )

    combined_df[f'mmCen_{name}'] = (
        combined_df[f'maxnorm_{name}']
        - combined_df[f'mean_{name}']
    )

    # --------------------------------------------------------
    # Original bin-level normalization
    # --------------------------------------------------------

    combined_df[f'maxbinnorm_{name}'] = (
        combined_df.groupby('label1')[
            f'{name}_bin'
        ].transform(group_minmax)
    )

    meanv_bin = (
        combined_df.groupby('label1')[
            f'maxbinnorm_{name}'
        ].mean()
    )

    combined_df[f'meanbin_{name}'] = (
        combined_df['label1'].map(meanv_bin)
    )

    combined_df[f'mmCenbin_{name}'] = (
        combined_df[f'maxbinnorm_{name}']
        - combined_df[f'meanbin_{name}']
    )

    # --------------------------------------------------------
    # Original optional normalization methods
    # --------------------------------------------------------

    if norm_method == 'relative':

        combined_df[f'relative_{name}'] = (
            combined_df[name]
            / combined_df['pixel_number']
        )

    elif norm_method == 'normalized':

        combined_df[f'Normalized_{name}'] = (
            combined_df[name]
            / (
                combined_df['total_int']
                / combined_df['pixel_number']
            )
        )

    elif norm_method == 'zscore':

        combined_df[f'zscore_{name}'] = (
            combined_df.groupby('label1')[name]
            .transform(zscore)
        )

    elif norm_method == 'minmax':

        combined_df[f'minmax_{name}'] = (
            combined_df.groupby('label1')[name]
            .transform(group_minmax)
        )

    combined_df = combined_df.reset_index(drop=True)

    return combined_df


# ------------------------------------------------------------
# 0.4 Import all 3 experiments and 3 groups
# ------------------------------------------------------------

usc = {}

for group in groups:
    usc[group] = []

for exp in experiments:

    print(f'\nProcessing {exp} ...')

    for group in groups:

        import_path = os.path.join(
            root_path,
            exp,
            group
        )

        if not os.path.isdir(import_path):
            raise FileNotFoundError(
                f'Folder not found: {import_path}'
            )

        combined_df = read_csv(
            folder_path=import_path,
            name='syn',
            norm_method='normalized',
            experiment_id=exp,
            group_name=group
        )

        n_cells = combined_df['Cell_ID'].nunique()

        print(
            f'{exp} | {group}: '
            f'{n_cells} cells'
        )

        if n_cells != expected_cells:
            raise ValueError(
                f'{exp}/{group}: Expected '
                f'{expected_cells} cells, '
                f'but found {n_cells}. '
                f'Please check the CSV files.'
            )

        usc[group].append(combined_df)


# ------------------------------------------------------------
# 0.5 Merge the three independent experiments
#     within each experimental group
# ------------------------------------------------------------

s = pd.concat(
    usc['SNCA'],
    ignore_index=True
)

u = pd.concat(
    usc['UBL3'],
    ignore_index=True
)

c = pd.concat(
    usc['CAAX'],
    ignore_index=True
)


# ------------------------------------------------------------
# 0.6 Combine all cells into one master DataFrame
# ------------------------------------------------------------

all_cells = pd.concat(
    [s, u, c],
    ignore_index=True
)


# ------------------------------------------------------------
# 0.7 Check experimental design
# ------------------------------------------------------------

print('\n========== Experimental design check ==========')

print(
    all_cells.groupby(
        ['Experiment_ID', 'Group']
    )['Cell_ID'].nunique()
)

print('\nCells per group:')
print(
    all_cells.groupby('Group')['Cell_ID'].nunique()
)

print('\nCells per experiment:')
print(
    all_cells.groupby('Experiment_ID')['Cell_ID'].nunique()
)

print('\nTotal unique cells:')
print(all_cells['Cell_ID'].nunique())

assert all_cells['Cell_ID'].nunique() == 90
assert s['Cell_ID'].nunique() == 30
assert u['Cell_ID'].nunique() == 30
assert c['Cell_ID'].nunique() == 30

print('\nCell 0 completed successfully.')
